### Parameters

In [0]:
# catalog
catalog = "workspace"

# cdc column
cdc_col = "modifiedDate"

# backdated refresh
backdated_refresh = ""

# source object
source_object = "silver_bookings"

# source schema
source_schema = "silver"

# source fact table
fact_table = f"{catalog}.{source_schema}.{source_object}"

# target schema
target_schema = "gold"

# target object
target_object = "factbookings"

# fact key column list
fact_key_cols = ['dimpassengerskey','dimflightskey','dimairportskey','booking_date']

In [0]:
dimensions = [
    {
        "table": "workspace.gold.dimpassengers",
        "alias": "dimpassengers",
        "join_key": [("passenger_id", "passenger_id")]  # (fact col, dim col)
    },
    {
        "table": "workspace.gold.dimflights",
        "alias": "dimflights",
        "join_key": [("flight_id", "flight_id")]  # (fact col, dim col)
    },
    {
        "table": "workspace.gold.dimairports",
        "alias": "dimairports",
        "join_key": [("airport_id", "airport_id")]  # (fact col, dim col)
    }   
]

# columns that we want to keep from the fact table (beside the surrogate keys)
fact_columns = ["amount","booking_date","modifiedDate"]

### last load date

In [0]:
# no backdated refresh
if len(backdated_refresh) == 0:
    
    # if table exists in the destination
    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

        last_load = spark.sql(f"SELECT max({cdc_col}) FROM {catalog}.{target_schema}.{target_object}").collect()[0][0]
    else:

        last_load = "1900-01-01 00:00:00"
# yes the backdated_refresh
else:
    last_load = backdated_refresh

# test the load date
last_load

### dynamic fact query (bring keys)

In [0]:
def generate_fact_query_incremental(fact_table,dimensions,fact_columns,cdc_col,processing_date):
    fact_alias ="f"

    # base column to select
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    # build join dynamically
    join_clauses = []
    for dim in dimensions:
        table_full = dim["table"]
        alias = dim["alias"]
        table_name = table_full.split('.')[-1]
        surrogate_key = f"{alias}.{table_name}key"
        select_cols.append(surrogate_key)

    # build ON clause
        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{dk}" for fk,dk in dim["join_key"]
                   ]
        join_clause = f"LEFT JOIN {table_full} {alias} ON " + "AND ".join(on_conditions)
        join_clauses.append(join_clause)

    # final select and join clauses
    select_clause = ",\n   ".join(select_cols)
    joins = "\n".join(join_clauses)

    # where clause for incremental filtering
    where_clause = f"{fact_alias}.{cdc_col} >= date('{processing_date}')"

    # final query
    query = f""" select {select_clause}
from {fact_table} {fact_alias}
{joins}
where {where_clause}
""".strip()

    return query



In [0]:
query = generate_fact_query_incremental(fact_table,dimensions,fact_columns,cdc_col,last_load)
print(query)

In [0]:
df_fact = spark.sql(query)

In [0]:
display(df_fact)

### upsert conditions

In [0]:
from delta.tables import DeltaTable

In [0]:
fact_key_cols_str = " AND ".join([f"src.{col} = trg.{col}" for col in fact_key_cols])
fact_key_cols_str

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_fact.alias("src"), fact_key_cols_str)\
            .whenMatchedUpdateAll(condition = f'src.{cdc_col} >= trg.{cdc_col}')\
            .whenNotMatchedInsertAll()\
            .execute()

else:
    df_fact.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")

In [0]:
%sql
select * from workspace.gold.factbookings

In [0]:
# df = spark.sql("select * from workspace.gold.dimairports").groupBy("dimairportskey").count().filter("count > 1")
# display(df)
# # I am checking the dimension table if there is any duplicacy